In [ ]:
import requests
import pandas as pd
import numpy as np  
import time
import ast

# 1. 인증 및 헤더 설정
headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'origin': 'https://www.yogiyo.co.kr',
    'referer': 'https://www.yogiyo.co.kr/',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
    'x-api-key': 'qua9EeW1ohth4ain',
    'x-ygy-app-version': '9.0.0',
    'x-ygy-os-type': 'IOS',
    'x-ygy-route': 'v2',
}

# 2. 파라미터 초기 설정
params = {
    'adm_code': '1162069500',
    'customer_id': '923892007',
    'lat': '37.4869016',
    'lng': '126.92737994',
    'length': '60',
    'sort': 'RANK_DESC',
    'vertical_types': 'FOOD',
    'membership_code': 'NONE',
    'serving_types': 'VD',
    'use_bargainyo': 'false'
}

url = 'https://api.yogiyo.co.kr/shopyo/v1/shops'
all_shops = []
start_val = 0
is_end = False

print("====== 1. 데이터 수집 시작 ======")

# 3. 데이터 수집 반복 루프
while not is_end:
    params['start'] = str(start_val)
    
    try:
        response = requests.get(url, params=params, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            shops_list = data.get('shops', [])
            
            if not shops_list:
                print(f"\n수집 완료: {start_val} 지점에서 데이터가 더 이상 없습니다.")
                is_end = True
                break
                
            all_shops.extend(shops_list)
            print(f"현재 위치: {start_val} / 누적 상점 수: {len(all_shops)}", end='\r')
            start_val += 60
            
        elif response.status_code in [401, 403]:
            print("\n인증 키 만료 혹은 접근 차단이 발생했습니다.")
            break
        else:
            print(f"\n오류 발생 (상태 코드: {response.status_code})")
            break
            
    except Exception as e:
        print(f"\n네트워크 예외 발생: {e}")
        break
        
    time.sleep(1.5)  # 서버 부하 방지

# 4. 수집 데이터 기본 전처리 및 가공
if all_shops:
    raw_df = pd.DataFrame(all_shops)
    print(f"\n총 {len(raw_df)}개의 상점 수집 완료. 전처리를 시작합니다...")
    
    # 필요한 주요 컬럼만 추출
    cols = ["name", "location", "vendor_categories", "review"]
    target_cols = [c for c in cols if c in raw_df.columns]
    new_df = raw_df[target_cols].copy()
    
    # 문자열로 된 딕셔너리가 있다면 파싱 (수집 직후엔 대개 dict 형태이나 안전장치로 유지)
    new_df['location'] = new_df['location'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    new_df['review'] = new_df['review'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    # 딕셔너리 내부 키들을 별도 컬럼으로 확장
    review_cols = new_df['review'].apply(pd.Series)
    location_cols = new_df['location'].apply(pd.Series)
    
    # 데이터프레임 결합 (기존 원본을 유지하면서 새 컬럼 우측 결합)
    processed_df = pd.concat([new_df, review_cols, location_cols], axis=1)
    print("기본 전처리 및 컬럼 확장 완료.")
else:
    print("\n수집된 데이터가 없어 프로세스를 종료합니다.")
    processed_df = pd.DataFrame()

====== 1. 데이터 수집 시작 ======
현재 위치: 1860 / 누적 상점 수: 1892
수집 완료: 1920 지점에서 데이터가 더 이상 없습니다.

총 1892개의 상점 수집 완료. 전처리를 시작합니다...
기본 전처리 및 컬럼 확장 완료.


In [3]:
from tqdm import tqdm
import numpy as np  # 셀 단독 실행 시 에러 방지용

if not processed_df.empty:
    print("\n====== 2. 주소 변환 및 매장 분석 시작 ======")
    
    # 1. 카카오 API 설정
    KAKAO_API_KEY = 'f62a560324bef030e4a4ef2a1ba39808'
    
    def get_full_address(lat, lng):
        url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        params = {"x": lng, "y": lat}  # 문자열 그대로 전달하여 정밀도 유지
        
        try:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code == 200:
                data = response.json()
                if data['documents']:
                    addr_info = data['documents'][0]
                    
                    road = addr_info.get('road_address')
                    lot = addr_info.get('address')
                    
                    full_addr = road['address_name'] if road else lot['address_name']
                    
                    return {
                        'full_address': full_addr,
                        'sido': lot['region_1depth_name'] if lot else None,
                        'gu': lot['region_2depth_name'] if lot else None,
                        'dong': lot['region_3depth_name'] if lot else None
                    }
        except:
            pass
        return {'full_address': None, 'sido': None, 'gu': None, 'dong': None}

    # 2. 전체 데이터 주소 변환 실행
    results = []
    for index, row in tqdm(processed_df.iterrows(), total=len(processed_df), desc="주소 변환 진행 중"):
        addr_data = get_full_address(row['lat'], row['lng'])
        results.append(addr_data)
        time.sleep(0.05)
        
    # 3. 주소 변환 결과 결합 (필터링 없이 기존 변수명 final_df 그대로 분석 진행)
    addr_df = pd.DataFrame(results)
    final_df = pd.concat([processed_df.reset_index(drop=True), addr_df], axis=1)
    
    print(f"\n주소 변환 완료 (총 {len(final_df)}건). 매장 유형 분류를 시작합니다...")
    
    # 4. 위경도 기반 공유주방/샵인샵 분석
    # 동일 좌표 내 매장 수 구하기
    final_df['shared_location_count'] = final_df.groupby(['lat', 'lng'])['name'].transform('size')
    
    # 분류 조건 및 라벨 설정
    conditions = [
        (final_df['shared_location_count'] == 1),
        (final_df['shared_location_count'] >= 2) & (final_df['shared_location_count'] <= 3),
        (final_df['shared_location_count'] >= 4)
    ]
    choices = ['독립 매장', '소규모 샵인샵', '대규모 공유주방']
    
    # store_type 컬럼 생성
    final_df['store_type'] = np.select(conditions, choices, default='분류 불가')
    
    # 5. 결과 데이터 검증 출력
    print("\n" + "="*5 + " 매장 타입 분류 결과 요약 " + "="*5)
    print(final_df['store_type'].value_counts())
    
    print("\n" + "="*5 + " 매장 타입별 평균 평점 " + "="*5)
    print(final_df.groupby('store_type')['average_rating'].mean())
    
    # 6. 태블로 전용 최종 파일로 저장
    final_df.to_csv('yogiyo_tableau_final.csv', index=False, encoding='utf-8-sig')
    print("\n🎉 태블로용 최종 파일 'yogiyo_tableau_final.csv' 저장 완료!")
else:
    print("가공할 데이터가 없습니다.")


====== 2. 주소 변환 및 매장 분석 시작 ======


주소 변환 진행 중: 100%|██████████| 1892/1892 [03:45<00:00,  8.39it/s]


주소 변환 완료 (총 1892건). 매장 유형 분류를 시작합니다...

===== 매장 타입 분류 결과 요약 =====
독립 매장       967
대규모 공유주방    479
소규모 샵인샵     446
Name: store_type, dtype: int64

===== 매장 타입별 평균 평점 =====
store_type
대규모 공유주방    4.362004
독립 매장       4.486556
소규모 샵인샵     4.455157
Name: average_rating, dtype: float64

🎉 태블로용 최종 파일 'yogiyo_tableau_final.csv' 저장 완료!


In [15]:
import pandas as pd


df = pd.read_csv('yogiyo_tableau_final.csv',encoding='utf-8')

In [16]:
df["Rank"]= df['count'].rank(ascending=False, method='min').astype(int)

In [7]:
df.to_csv('yogiyo_tableau_final2.csv', index=False, encoding='utf-8-sig')

In [18]:
df_gwanak = df[df['gu'] == '관악구'].copy()

In [21]:
df_gwanak['review_rank'] = df_gwanak['count'].rank(ascending=False, method='min')

In [23]:
df_gwanak.to_csv('yogiyo_tableau_final2.csv', index=False, encoding='utf-8-sig')

In [22]:
df_gwanak

,name,location,vendor_categories,review,average_rating,count,image_count,reply_count,lat,lng,full_address,sido,gu,dong,shared_location_count,store_type,Rank,review_rank
0,대치동엄마도시락-본점,"{'lat': '37.4839342', 'lng': '126.93936786'}","['찜/탕', '한식', '도시락/죽', '포장']","{'average_rating': 4.9, 'count': 195, 'image_c...",4.9,195,157,179,37.483934,126.939368,서울특별시 관악구 봉천로 356,서울,관악구,봉천동,6,대규모 공유주방,677,462.0
1,앵그리포테이토치킨&버거-신림점,"{'lat': '37.4837321', 'lng': '126.92774951'}","['피자/양식', '버거', '분식', '포장', '1인분주문']","{'average_rating': 5.0, 'count': 203, 'image_c...",5.0,203,205,43,37.483732,126.927750,서울특별시 관악구 남부순환로 1594,서울,관악구,신림동,1,독립 매장,666,455.0
2,호호솥밥-관악점,"{'lat': '37.4854819', 'lng': '126.93790844'}","['한식', '고기/구이', '일식/돈까스', '포장', '프랜차이즈', '신규맛집']","{'average_rating': 5.0, 'count': 20, 'image_co...",5.0,20,20,0,37.485482,126.937908,서울특별시 관악구 봉천로25길 4,서울,관악구,봉천동,2,소규모 샵인샵,1310,878.0
3,육회한날연어어때,"{'lat': '37.48203242479008', 'lng': '126.92966...","['포장', '일식/돈까스', '야식', '회/초밥']","{'average_rating': 5.0, 'count': 15593, 'image...",5.0,15593,14295,13506,37.482032,126.929660,서울특별시 관악구 신림로 309,서울,관악구,신림동,4,대규모 공유주방,4,1.0
4,괴짜반점-본점,"{'lat': '37.4839734', 'lng': '126.93930939'}","['한식', '중국집', '야식', '찜/탕', '포장']","{'average_rating': 4.8, 'count': 3847, 'image_...",4.8,3847,3532,2868,37.483973,126.939309,서울특별시 관악구 봉천로 356,서울,관악구,봉천동,1,독립 매장,67,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1833,가마솥도시락&일식,"{'lat': '37.4686969991815', 'lng': '126.940417...","['한식', '일식/돈까스', '치킨']","{'average_rating': 4.5, 'count': 8, 'image_cou...",4.5,8,2,7,37.468697,126.940417,서울특별시 관악구 대학7길 49,서울,관악구,신림동,5,대규모 공유주방,1461,979.0
1835,몬스터탕수육-낙성대점,"{'lat': '37.4775878', 'lng': '126.9628786'}","['중국집', '프랜차이즈', '포장']","{'average_rating': 4.9, 'count': 119, 'image_c...",4.9,119,112,36,37.477588,126.962879,서울특별시 관악구 남부순환로 1921-1,서울,관악구,봉천동,1,독립 매장,824,557.0
1836,냥빵,"{'lat': '37.4830036', 'lng': '126.91210288'}","['카페/디저트', '샌드위치', '포장', '샐러드']","{'average_rating': 4.9, 'count': 587, 'image_c...",4.9,587,552,587,37.483004,126.912103,서울특별시 관악구 조원로16가길 26,서울,관악구,신림동,1,독립 매장,370,251.0
1866,파스타는보니따-낙성대역점,"{'lat': '37.4752917', 'lng': '126.96686734'}","['피자/양식', '아시안', '분식', '포장', '프랜차이즈']","{'average_rating': 0.0, 'count': 0, 'image_cou...",0.0,0,0,0,37.475292,126.966867,서울특별시 관악구 인헌길 6,서울,관악구,봉천동,1,독립 매장,1752,1178.0
